# Noice-Inspired Recommender Demo

This notebook gives a lightweight portfolio walkthrough of the Noice-inspired audio recommendation system. It uses public catalog-style metadata and synthetic listening events only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_pipeline import preprocess
from src.recommender import build_recommender

preprocess()
content = pd.read_csv(PROJECT_ROOT / 'data/processed/content_processed.csv')
interactions = pd.read_csv(PROJECT_ROOT / 'data/processed/interactions_processed.csv')

## Dataset Size

Quick check of processed catalog items and synthetic interaction events.

In [ ]:
summary = {
    'content_items': len(content),
    'synthetic_interactions': len(interactions),
    'users': interactions['user_id'].nunique(),
    'genres': content['genre'].nunique(),
}
summary

## Genre Distribution

This shows the mix of audio categories available in the processed catalog.

In [ ]:
content['genre'].value_counts()

## Event Type Distribution

Synthetic event types are converted into implicit feedback scores for training.

In [ ]:
interactions['event_type'].value_counts()

## Personalized Recommendation Example

Recommendations for a known synthetic user use the hybrid model: item-based collaborative filtering plus TF-IDF content similarity.

In [ ]:
recommender = build_recommender()
pd.DataFrame([item.to_dict() for item in recommender.recommend_for_user('u_001', top_k=5)])

## Cold-Start Fallback

Unknown users do not have listening history, so the system falls back to trending/popular content based on aggregate implicit feedback.

In [ ]:
pd.DataFrame([item.to_dict() for item in recommender.recommend_for_user('new_user_001', top_k=5)])